# 01 — SDK-Sovereign Smoke Test

**Purpose:** Load Qwen 2.5-0.5B with two LoRA adapters, run a smoke episode against the live HF Space env, verify adapter swap.

**Runtime:** Colab T4 GPU (free tier) · ~10 min

> Before running: set `HF_TOKEN` in **Colab Secrets** (key icon on the left sidebar)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub
print('✓ Dependencies installed')

In [ ]:
# Cell 2 — Clone and install the SDK-Sovereign env package
import os, sys

HF_USER   = 'ishansurdi'
SPACE_REPO = 'ishansurdi/SDK-Sovereign'
ENV_URL    = 'https://ishansurdi-sdk-sovereign.hf.space'

!git clone https://huggingface.co/spaces/{SPACE_REPO} sdk_sovereign_pkg
!pip install -q -e sdk_sovereign_pkg/
sys.path.insert(0, 'sdk_sovereign_pkg')
print(f'✓ Env package installed from {SPACE_REPO}')

In [ ]:
# Cell 3 — HF auth
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('✓ Logged in to HF Hub')

In [ ]:
# Cell 4 — Load Qwen 2.5-0.5B with two LoRA adapters
from server.llm_agents import load_model_with_two_adapters, make_agent_pair

model, tokenizer = load_model_with_two_adapters()
print('Adapters:', list(model.peft_config.keys()))
model.print_trainable_parameters()

agents = make_agent_pair(model, tokenizer)
print('✓ Agent pair created')

In [ ]:
# Cell 5 — Run smoke episode against live HF Space
from client import SDKSovereignEnv

print(f'Connecting to: {ENV_URL}')

with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
    obs = env.reset()
    print(f'Episode started | role={obs.current_role}')
    for turn in range(7):
        agent = agents[obs.current_role]
        action = agent(obs)
        print(f'Turn {turn:2d} | {obs.current_role:7s} | {action.action_type}')
        obs = env.step(action)
        if obs.done:
            print(f'Done! reward={obs.reward:.2f} | breakdown={obs.reward_breakdown}')
            break
print('✓ Smoke episode complete')

In [ ]:
# Cell 6 — Verify adapter swap
from models import SDKObservation

dummy_aud = SDKObservation(
    current_role='auditor', turn_index=0, max_turns=7, error_log='test',
    conversation_history=[], visible_allowlist=['razorpay'],
    visible_codebase=None, visible_filename=None, current_proposal=None,
    approved_replacement=None, done=False, reward=0.0, reward_breakdown={},
)
agents['auditor'](dummy_aud)
assert model.active_adapter == 'auditor_adapter'
print('✓ auditor_adapter swaps correctly')

dummy_lead = SDKObservation(
    current_role='lead', turn_index=1, max_turns=7, error_log='test',
    conversation_history=[], visible_allowlist=None,
    visible_codebase='import stripe', visible_filename='payment.py',
    current_proposal=None, approved_replacement=None,
    done=False, reward=0.0, reward_breakdown={},
)
agents['lead'](dummy_lead)
assert model.active_adapter == 'lead_adapter'
print('✓ lead_adapter swaps correctly')
print('\n✅ Phase 6 acceptance criteria met!')